# Gold - Deteccion de fraude

Parte 4 del proyecto: leer Silver, aplicar reglas de riesgo, generar `fraud_alerts` y construir una tabla tabular para analisis de relaciones.

In [50]:
import os
import socket
import time
from urllib import error, request
from py4j.protocol import Py4JJavaError
from pyspark.sql import SparkSession


def resolve_service_host(service_name: str, fallback_host: str = "localhost") -> str:
    try:
        socket.gethostbyname(service_name)
        return service_name
    except socket.gaierror:
        return fallback_host


def is_http_ready(url: str, timeout: float = 2.0) -> bool:
    try:
        with request.urlopen(url, timeout=timeout) as _:
            return True
    except error.HTTPError:
        return True
    except Exception:
        return False


iceberg_host = resolve_service_host("iceberg-rest")
minio_host = resolve_service_host("minio")

default_iceberg_uri = f"http://{iceberg_host}:8181"
default_minio_endpoint = f"http://{minio_host}:{'9000' if minio_host == 'minio' else '9100'}"

ICEBERG_REST_URI = os.getenv("ICEBERG_REST_URI", default_iceberg_uri)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", default_minio_endpoint)

print(f"ICEBERG_REST_URI={ICEBERG_REST_URI}")
print(f"MINIO_ENDPOINT={MINIO_ENDPOINT}")

iceberg_config_url = f"{ICEBERG_REST_URI}/v1/config"
for _ in range(20):
    if is_http_ready(iceberg_config_url):
        break
    time.sleep(2)

spark = (
    SparkSession.builder
    .appName("gold-fraud-job")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
            "org.apache.iceberg:iceberg-aws-bundle:1.6.1"
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type", "rest")
    .config("spark.sql.catalog.lakehouse.uri", ICEBERG_REST_URI)
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint", MINIO_ENDPOINT)
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "admin")
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "admin123")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

for attempt in range(1, 11):
    try:
        spark.sql("CREATE NAMESPACE IF NOT EXISTS lakehouse.gold")
        print("Spark e Iceberg listos para Gold")
        break
    except Py4JJavaError as e:
        message = str(e)
        transient = "UnknownHostException" in message or "RESTException" in message
        if transient and attempt < 10:
            print(f"Reintento {attempt}/10: esperando catalogo Iceberg...")
            time.sleep(3)
            continue
        raise

ICEBERG_REST_URI=http://iceberg-rest:8181
MINIO_ENDPOINT=http://minio:9000
Spark e Iceberg listos para Gold


## 1) Leer Silver y preparar base

In [51]:
from pyspark.sql import functions as F

silver_df = spark.table("lakehouse.silver.payments_enriched")

base_df = silver_df.select(
    "payment_id",
    "event_ts",
    "customer_id",
    "card_id",
    "merchant_id",
    "device_id",
    "ip",
    "country",
    "amount",
    "currency",
    "status",
    "tx_count_card_5m",
    "distinct_merchants_card_10m",
    "distinct_countries_card_1h",
    "distinct_cards_device_1h",
    "declined_ratio_card_1h"
)

base_df.show(5, truncate=False)

+------------------------------------+--------------------------+-----------+----------+-----------+---------+---------------+-------+------+--------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|payment_id                          |event_ts                  |customer_id|card_id   |merchant_id|device_id|ip             |country|amount|currency|status  |tx_count_card_5m|distinct_merchants_card_10m|distinct_countries_card_1h|distinct_cards_device_1h|declined_ratio_card_1h|
+------------------------------------+--------------------------+-----------+----------+-----------+---------+---------------+-------+------+--------+--------+----------------+---------------------------+--------------------------+------------------------+----------------------+
|5c4d3bb8-b6c4-40a5-85d8-4c772977f277|2026-01-14 00:38:40.930214|CUST_00008 |CARD_00003|MERCH_00086|DEV_00017|26.144.238.219 |AO     |244.33|USD     |approved|1

## 2) Reglas de fraude, reasons y risk_score

In [52]:
ALERT_THRESHOLD = 25

rules_df = (
    silver_df
    .withColumn("rule_high_tx_velocity",     F.col("tx_count_card_5m") >= 12)
    .withColumn("rule_many_merchants",       F.col("distinct_merchants_card_10m") >= 12)
    .withColumn("rule_many_countries",       F.col("distinct_countries_card_1h") >= 8)
    .withColumn("rule_device_shared",        F.col("distinct_cards_device_1h") >= 16)
    .withColumn("rule_high_declined_ratio",  F.col("declined_ratio_card_1h") >= 0.65)
    .withColumn("rule_high_amount",          F.col("amount") >= 700.00)
)

scored_df = (
    rules_df
    .withColumn(
        "risk_score_raw",
        F.when(F.col("rule_high_tx_velocity"),    F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_many_merchants"),      F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_many_countries"),      F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_device_shared"),       F.lit(20)).otherwise(F.lit(0)) +
        F.when(F.col("rule_high_declined_ratio"), F.lit(25)).otherwise(F.lit(0)) +
        F.when(F.col("rule_high_amount"),         F.lit(15)).otherwise(F.lit(0))
    )
    .withColumn("risk_score", F.least(F.lit(100), F.col("risk_score_raw")).cast("int"))
    .withColumn(
        "is_fraud_alert",
        F.col("risk_score") >= ALERT_THRESHOLD
    )
    .withColumn(
        "reasons",
        F.expr("""
            filter(
                array(
                    IF(rule_high_tx_velocity,    'high_tx_velocity_5m', NULL),
                    IF(rule_many_merchants,      'many_merchants_10m', NULL),
                    IF(rule_many_countries,      'many_countries_1h', NULL),
                    IF(rule_device_shared,       'device_shared_1h', NULL),
                    IF(rule_high_declined_ratio, 'high_declined_ratio_1h', NULL),
                    IF(rule_high_amount,         'high_amount', NULL)
                ),
                x -> x IS NOT NULL
            )
        """)
    )
)


## 3) Tabla gold.fraud_alerts

In [53]:
fraud_alerts_df = (
    scored_df
    .filter(F.col("risk_score") >= ALERT_THRESHOLD)
    .select(
        "payment_id",
        "event_ts",
        "customer_id",
        "card_id",
        "merchant_id",
        "device_id",
        "ip",
        "country",
        "amount",
        "currency",
        "status",
        "tx_count_card_5m",
        "distinct_merchants_card_10m",
        "distinct_countries_card_1h",
        "distinct_cards_device_1h",
        "declined_ratio_card_1h",
        "risk_score",
        "reasons"
    )
)

(
    fraud_alerts_df.writeTo("lakehouse.gold.fraud_alerts")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print("Tabla Gold creada: lakehouse.gold.fraud_alerts")

Tabla Gold creada: lakehouse.gold.fraud_alerts


## 4) Tabla tabular para analisis de relaciones

In [54]:
relations_df = (
    scored_df
    .select(
        "payment_id",
        "event_ts",
        "customer_id",
        "card_id",
        "merchant_id",
        "device_id",
        "ip",
        "country",
        "amount",
        "currency",
        "status",
        "risk_score",
        "reasons"
    )
    .withColumn("is_alert", F.col("risk_score") >= ALERT_THRESHOLD)
)

(
    relations_df.writeTo("lakehouse.gold.payment_relations")
    .using("iceberg")
    .tableProperty("write.format.default", "parquet")
    .createOrReplace()
)

print("Tabla Gold creada: lakehouse.gold.payment_relations")

Tabla Gold creada: lakehouse.gold.payment_relations


## 5) Validaciones rapidas

In [55]:
spark.sql("SELECT COUNT(*) AS total_alerts FROM lakehouse.gold.fraud_alerts").show()
spark.sql("SELECT COUNT(*) AS total_relations FROM lakehouse.gold.payment_relations").show()

spark.sql("""
SELECT payment_id, event_ts, card_id, merchant_id, risk_score, reasons
FROM lakehouse.gold.fraud_alerts
ORDER BY risk_score DESC, event_ts DESC
LIMIT 10
""").show(truncate=False)

+------------+
|total_alerts|
+------------+
|        4092|
+------------+

+---------------+
|total_relations|
+---------------+
|          65960|
+---------------+

+--------------------------------------------+--------------------------+----------+-----------+----------+-------------------------------------+
|payment_id                                  |event_ts                  |card_id   |merchant_id|risk_score|reasons                              |
+--------------------------------------------+--------------------------+----------+-----------+----------+-------------------------------------+
|3ed981e3-4a56-4494-9cff-a02556d605ba        |2026-04-09 09:59:38.720971|CARD_00125|MERCH_00006|40        |[high_declined_ratio_1h, high_amount]|
|bd07d680-be5d-4460-bca5-ef764ff99219        |2026-04-06 20:16:19.3861  |CARD_00032|MERCH_00014|40        |[high_declined_ratio_1h, high_amount]|
|53773c5a-1492-4e55-af25-5cb7d8a64e53_retry_1|2026-04-02 00:24:33.354528|CARD_00008|MERCH_00039|40     

In [56]:
spark.sql("""
SELECT
  COUNT(*) AS total_alerts,
  MAX(risk_score) AS max_risk_score,
  ROUND(AVG(risk_score), 2) AS avg_risk_score
FROM lakehouse.gold.fraud_alerts
""").show()

spark.sql("""
SELECT COUNT(*) AS total_relations
FROM lakehouse.gold.payment_relations
""").show()

+------------+--------------+--------------+
|total_alerts|max_risk_score|avg_risk_score|
+------------+--------------+--------------+
|        4092|            40|         25.07|
+------------+--------------+--------------+

+---------------+
|total_relations|
+---------------+
|          65960|
+---------------+

